In [1]:

import pandas as pd
import sys
from pathlib import Path
from sqlalchemy import text

# Add the RetailIQ project root to Python's import path
PROJECT_DIR = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_DIR))

from database.db_connection import get_engine

engine = get_engine()

try:
    with engine.connect() as conn:
        # Load current product inventory
        products = pd.read_sql(
            text("""
                SELECT
                    Product_ID,
                    Product_Name,
                    Category,
                    Selling_Price,
                    Stock_Quantity
                FROM products
            """),
            conn
        )

        # Load delivered sales for demand analysis
        sales = pd.read_sql(
            text("""
                SELECT
                    Order_ID,
                    Product_ID,
                    Order_Date,
                    Quantity,
                    Order_Status
                FROM sales
                WHERE Order_Status = 'Delivered'
            """),
            conn
        )
finally:
    engine.dispose()

print("Products loaded:", len(products))
print("Delivered sales records:", len(sales))
print("Unique products sold:", sales["Product_ID"].nunique())

display(products.head())
display(sales.head())


Products loaded: 2000
Delivered sales records: 200139
Unique products sold: 2000


,Product_ID,Product_Name,Category,Selling_Price,Stock_Quantity
0,PROD000001,Samsung Mobile V5,Electronics,100532.23,159
1,PROD000002,HP Laptop V5,Electronics,24086.65,54
2,PROD000003,boAt Headphones V2,Electronics,61821.01,638
3,PROD000004,Noise Watch V7,Electronics,43075.89,572
4,PROD000005,Apple iPhone V4,Electronics,7466.64,865


,Order_ID,Product_ID,Order_Date,Quantity,Order_Status
0,ORD0000000001,PROD001017,2026-06-07,1,Delivered
1,ORD0000000002,PROD000703,2025-02-04,1,Delivered
2,ORD0000000004,PROD000213,2025-01-27,1,Delivered
3,ORD0000000005,PROD001437,2024-06-30,2,Delivered
4,ORD0000000008,PROD001392,2024-06-19,2,Delivered


In [2]:

# Ensure dates and quantities have the correct data types
sales["Order_Date"] = pd.to_datetime(sales["Order_Date"], errors="coerce")
sales["Quantity"] = pd.to_numeric(sales["Quantity"], errors="coerce")

# Use the latest date in the dataset as the reference date
reference_date = sales["Order_Date"].max()
start_date = reference_date - pd.Timedelta(days=29)

# Filter sales to the 30-day window, including both endpoints
recent_sales = sales[
    (sales["Order_Date"] >= start_date)
    & (sales["Order_Date"] <= reference_date)
].copy()

# Calculate total units sold for each product
recent_demand = (
    recent_sales.groupby("Product_ID")["Quantity"]
    .sum()
    .reset_index(name="Units_Sold_30_Days")
)

print("Reference date:", reference_date.date())
print("30-day period:", start_date.date(), "to", reference_date.date())
print("Sales records in period:", len(recent_sales))

display(
    recent_demand
    .sort_values("Units_Sold_30_Days", ascending=False)
    .head(10)
)


Reference date: 2026-06-30
30-day period: 2026-06-01 to 2026-06-30
Sales records in period: 7961


,Product_ID,Units_Sold_30_Days
836,PROD000878,22
734,PROD000772,20
1694,PROD001787,18
267,PROD000281,18
737,PROD000775,18
1693,PROD001786,17
1895,PROD001996,17
467,PROD000494,17
1431,PROD001509,17
533,PROD000562,16


In [3]:

# Combine current inventory with recent demand
inventory = products.merge(
    recent_demand,
    on="Product_ID",
    how="left"
)

# Products with no sales in the period get zero recent demand
inventory["Units_Sold_30_Days"] = (
    inventory["Units_Sold_30_Days"]
    .fillna(0)
)

# Calculate average daily demand
inventory["Average_Daily_Demand"] = (
    inventory["Units_Sold_30_Days"] / 30
)

# Estimate how many days current stock will last
inventory["Days_of_Stock_Left"] = (
    inventory["Stock_Quantity"]
    / inventory["Average_Daily_Demand"].replace(0, float("nan"))
)

# Assign restock priority using stock coverage
def assign_priority(row):
    if row["Average_Daily_Demand"] == 0:
        return "No Recent Sales"

    if row["Days_of_Stock_Left"] <= 7:
        return "High"

    if row["Days_of_Stock_Left"] <= 14:
        return "Medium"

    return "Low"

inventory["Restock_Priority"] = inventory.apply(
    assign_priority,
    axis=1
)

# Display products that need the most attention
priority_order = {
    "High": 0,
    "Medium": 1,
    "Low": 2,
    "No Recent Sales": 3
}

inventory["_Priority_Order"] = inventory["Restock_Priority"].map(
    priority_order
)

inventory = inventory.sort_values(
    ["_Priority_Order", "Days_of_Stock_Left"],
    ascending=[True, True],
    na_position="last"
).drop(columns="_Priority_Order")

print("Inventory analysis completed.")

display(
    inventory[
        [
            "Product_ID",
            "Product_Name",
            "Stock_Quantity",
            "Units_Sold_30_Days",
            "Average_Daily_Demand",
            "Days_of_Stock_Left",
            "Restock_Priority"
        ]
    ].head(20)
)


Inventory analysis completed.


,Product_ID,Product_Name,Stock_Quantity,Units_Sold_30_Days,Average_Daily_Demand,Days_of_Stock_Left,Restock_Priority
1998,PROD001999,Noise Watch V2,15,16.0,0.533333,28.125000,Low
1957,PROD001958,Lays Chips V6,10,8.0,0.266667,37.500000,Low
735,PROD000736,Samsung Mobile V2,14,11.0,0.366667,38.181818,Low
1405,PROD001406,Levis Jeans V6,12,9.0,0.300000,40.000000,Low
1822,PROD001823,boAt Headphones V5,15,9.0,0.300000,50.000000,Low
1618,PROD001619,Titan Watch V9,19,10.0,0.333333,57.000000,Low
1820,PROD001821,Samsung Mobile V2,27,14.0,0.466667,57.857143,Low
137,PROD000138,Lays Chips V8,13,6.0,0.200000,65.000000,Low
1596,PROD001597,Puma Shoes V9,13,6.0,0.200000,65.000000,Low
792,PROD000793,Decathlon Cycle V8,22,10.0,0.333333,66.000000,Low


In [4]:
# Count products in each restock priority
print("Restock priority distribution:")
display(
    inventory["Restock_Priority"]
    .value_counts()
    .rename_axis("Priority")
    .reset_index(name="Product_Count")
)

# Show products that need attention first
display(
    inventory[
        inventory["Restock_Priority"].isin(["High", "Medium"])
    ][
        [
            "Product_ID",
            "Product_Name",
            "Stock_Quantity",
            "Units_Sold_30_Days",
            "Average_Daily_Demand",
            "Days_of_Stock_Left",
            "Restock_Priority"
        ]
    ].sort_values("Days_of_Stock_Left")
    .head(20)
)

Restock priority distribution:


,Priority,Product_Count
0,Low,1900
1,No Recent Sales,100


,Product_ID,Product_Name,Stock_Quantity,Units_Sold_30_Days,Average_Daily_Demand,Days_of_Stock_Left,Restock_Priority


In [5]:
# Show products with the lowest stock coverage
display(
    inventory[
        inventory["Average_Daily_Demand"] > 0
    ][
        [
            "Product_ID",
            "Product_Name",
            "Stock_Quantity",
            "Units_Sold_30_Days",
            "Average_Daily_Demand",
            "Days_of_Stock_Left"
        ]
    ]
    .sort_values("Days_of_Stock_Left")
    .head(15)
    .round(2)
)

,Product_ID,Product_Name,Stock_Quantity,Units_Sold_30_Days,Average_Daily_Demand,Days_of_Stock_Left
1998,PROD001999,Noise Watch V2,15,16.0,0.53,28.12
1957,PROD001958,Lays Chips V6,10,8.0,0.27,37.50
735,PROD000736,Samsung Mobile V2,14,11.0,0.37,38.18
1405,PROD001406,Levis Jeans V6,12,9.0,0.30,40.00
1822,PROD001823,boAt Headphones V5,15,9.0,0.30,50.00
1618,PROD001619,Titan Watch V9,19,10.0,0.33,57.00
1820,PROD001821,Samsung Mobile V2,27,14.0,0.47,57.86
137,PROD000138,Lays Chips V8,13,6.0,0.20,65.00
1596,PROD001597,Puma Shoes V9,13,6.0,0.20,65.00
792,PROD000793,Decathlon Cycle V8,22,10.0,0.33,66.00


In [6]:

# Define the target inventory coverage in days
TARGET_STOCK_DAYS = 30

# Calculate the desired stock level
inventory["Target_Stock_Quantity"] = (
    inventory["Average_Daily_Demand"] * TARGET_STOCK_DAYS
).apply(__import__("math").ceil)

# Calculate how many units to replenish
inventory["Suggested_Restock_Quantity"] = (
    inventory["Target_Stock_Quantity"] - inventory["Stock_Quantity"]
).clip(lower=0)

# Assign a purchasing recommendation
def get_stock_action(row):
    if row["Units_Sold_30_Days"] == 0:
        return "Review Demand"
    elif row["Suggested_Restock_Quantity"] > 0:
        return "Consider Restocking"
    else:
        return "Stock Sufficient"

inventory["Stock_Action"] = inventory.apply(
    get_stock_action,
    axis=1
)

# Display products that may need replenishment
restock_view = inventory[
    inventory["Suggested_Restock_Quantity"] > 0
].sort_values(
    "Suggested_Restock_Quantity",
    ascending=False
)

print("Products suggested for restocking:", len(restock_view))

display(
    restock_view[
        [
            "Product_ID",
            "Product_Name",
            "Stock_Quantity",
            "Units_Sold_30_Days",
            "Days_of_Stock_Left",
            "Target_Stock_Quantity",
            "Suggested_Restock_Quantity",
            "Stock_Action"
        ]
    ].head(20).round(2)
)


Products suggested for restocking: 1


,Product_ID,Product_Name,Stock_Quantity,Units_Sold_30_Days,Days_of_Stock_Left,Target_Stock_Quantity,Suggested_Restock_Quantity,Stock_Action
1998,PROD001999,Noise Watch V2,15,16.0,28.12,16,1,Consider Restocking


In [7]:
# Summarize inventory recommendations
print("Total products analyzed:", len(inventory))

print("\nStock action summary:")
display(
    inventory["Stock_Action"]
    .value_counts()
    .rename_axis("Stock_Action")
    .reset_index(name="Product_Count")
)

print("\nTotal units suggested for restocking:",
      int(inventory["Suggested_Restock_Quantity"].sum()))

print("\nProducts requiring demand review:",
      int((inventory["Stock_Action"] == "Review Demand").sum()))

Total products analyzed: 2000

Stock action summary:


,Stock_Action,Product_Count
0,Stock Sufficient,1899
1,Review Demand,100
2,Consider Restocking,1



Total units suggested for restocking: 1

Products requiring demand review: 100
